# NASA PCoE Lithium-Ion Battery Dataset: Feature Extraction and Preprocessing Pipeline

## Multi-Cell Degradation Telemetry Analysis (B0005, B0006, B0007, B0018)

This notebook processes raw cycling telemetry from the official **NASA Ames Prognostics Center of Excellence (PCoE)** battery aging repository. The pipeline parses multi-cycle charge, discharge, and electrochemical impedance spectroscopy (EIS) measurements across four commercial 18650 Li-ion cells, extracts physical degradation indicators, computes State of Health (SOH) and Remaining Useful Life (RUL) targets, and standardizes features for predictive modeling.

## Environment Setup and Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["figure.dpi"] = 120

print("Environment initialized successfully.")

## Module 1: Telemetry Feature Extraction Engine

Extracts time-series summaries across discharge curves (voltage, current, temperature, dwell durations), charging profiles (CC and CV times), and electrochemical impedance checks (electrolyte resistance $R_e$, charge transfer resistance $R_{ct}$).

In [ ]:
def extract_battery_data(mat_path, battery_id, nominal_capacity=2.0, eol_threshold=1.40):
    mat = sio.loadmat(mat_path)
    battery = mat[battery_id][0, 0]
    cycles = battery['cycle'][0]
    
    last_Re = np.nan
    last_Rct = np.nan
    
    charge_info = {}
    charge_cycle_idx = 0
    
    for c in cycles:
        t = c['type'][0]
        d = c['data'][0, 0]
        
        if t == 'impedance':
            if 'Re' in d.dtype.names and len(d['Re']) > 0:
                last_Re = float(d['Re'][0][0])
            if 'Rct' in d.dtype.names and len(d['Rct']) > 0:
                last_Rct = float(d['Rct'][0][0])
                
        elif t == 'charge':
            charge_cycle_idx += 1
            v_meas = d['Voltage_measured'][0] if 'Voltage_measured' in d.dtype.names else []
            c_meas = d['Current_measured'][0] if 'Current_measured' in d.dtype.names else []
            temp_meas = d['Temperature_measured'][0] if 'Temperature_measured' in d.dtype.names else []
            t_meas = d['Time'][0] if 'Time' in d.dtype.names else []
            
            cc_time = np.nan
            cv_time = np.nan
            total_chg_time = np.nan
            max_chg_temp = np.nan
            
            if len(v_meas) > 0 and len(t_meas) > 0:
                total_chg_time = float(t_meas[-1] - t_meas[0])
                cc_indices = np.where(v_meas >= 4.19)[0]
                if len(cc_indices) > 0:
                    cc_time = float(t_meas[cc_indices[0]])
                    cv_time = float(total_chg_time - cc_time)
                else:
                    cc_time = total_chg_time
                    cv_time = 0.0
            if len(temp_meas) > 0:
                max_chg_temp = float(np.max(temp_meas))
                
            charge_info[charge_cycle_idx] = {
                'CC_Charge_Time_s': cc_time,
                'CV_Charge_Time_s': cv_time,
                'Total_Charge_Time_s': total_chg_time,
                'Max_Charge_Temp_C': max_chg_temp
            }
            
    records = []
    dis_cycle_idx = 0
    
    for c in cycles:
        t = c['type'][0]
        if t != 'discharge':
            continue
            
        dis_cycle_idx += 1
        d = c['data'][0, 0]
        
        cap = float(d['Capacity'][0][0]) if ('Capacity' in d.dtype.names and len(d['Capacity']) > 0) else np.nan
        v_meas = d['Voltage_measured'][0] if 'Voltage_measured' in d.dtype.names else []
        c_meas = d['Current_measured'][0] if 'Current_measured' in d.dtype.names else []
        temp_meas = d['Temperature_measured'][0] if 'Temperature_measured' in d.dtype.names else []
        t_meas = d['Time'][0] if 'Time' in d.dtype.names else []
        
        v_max = float(np.max(v_meas)) if len(v_meas) > 0 else np.nan
        v_min = float(np.min(v_meas)) if len(v_meas) > 0 else np.nan
        v_mean = float(np.mean(v_meas)) if len(v_meas) > 0 else np.nan
        c_mean = float(np.mean(c_meas)) if len(c_meas) > 0 else np.nan
        temp_max = float(np.max(temp_meas)) if len(temp_meas) > 0 else np.nan
        temp_mean = float(np.mean(temp_meas)) if len(temp_meas) > 0 else np.nan
        dis_duration = float(t_meas[-1] - t_meas[0]) if len(t_meas) > 0 else np.nan
        
        t_to_3_5V = np.nan
        t_to_3_2V = np.nan
        if len(v_meas) > 0 and len(t_meas) > 0:
            idx_35 = np.where(v_meas <= 3.5)[0]
            if len(idx_35) > 0:
                t_to_3_5V = float(t_meas[idx_35[0]])
            idx_32 = np.where(v_meas <= 3.2)[0]
            if len(idx_32) > 0:
                t_to_3_2V = float(t_meas[idx_32[0]])
                
        chg = charge_info.get(dis_cycle_idx, {
            'CC_Charge_Time_s': np.nan,
            'CV_Charge_Time_s': np.nan,
            'Total_Charge_Time_s': np.nan,
            'Max_Charge_Temp_C': np.nan
        })
        
        records.append({
            'Battery_ID': battery_id,
            'Cycle_Index': dis_cycle_idx,
            'Discharge_Capacity_Ah': cap,
            'SOH_Pct': (cap / nominal_capacity) * 100.0 if not np.isnan(cap) else np.nan,
            'Max_Discharge_Voltage_V': v_max,
            'Min_Discharge_Voltage_V': v_min,
            'Mean_Discharge_Voltage_V': v_mean,
            'Mean_Discharge_Current_A': c_mean,
            'Max_Discharge_Temp_C': temp_max,
            'Mean_Discharge_Temp_C': temp_mean,
            'Discharge_Duration_s': dis_duration,
            'Time_to_3_5V_s': t_to_3_5V,
            'Time_to_3_2V_s': t_to_3_2V,
            'CC_Charge_Time_s': chg['CC_Charge_Time_s'],
            'CV_Charge_Time_s': chg['CV_Charge_Time_s'],
            'Total_Charge_Time_s': chg['Total_Charge_Time_s'],
            'Max_Charge_Temp_C': chg['Max_Charge_Temp_C'],
            'Electrolyte_Resistance_Re': last_Re,
            'Charge_Transfer_Resistance_Rct': last_Rct
        })
        
    df_bat = pd.DataFrame(records)
    
    # Missing value imputation
    df_bat['Electrolyte_Resistance_Re'] = df_bat['Electrolyte_Resistance_Re'].bfill().ffill()
    df_bat['Charge_Transfer_Resistance_Rct'] = df_bat['Charge_Transfer_Resistance_Rct'].bfill().ffill()
    df_bat['Max_Charge_Temp_C'] = df_bat['Max_Charge_Temp_C'].bfill().ffill()
    df_bat['CC_Charge_Time_s'] = df_bat['CC_Charge_Time_s'].bfill().ffill()
    df_bat['CV_Charge_Time_s'] = df_bat['CV_Charge_Time_s'].bfill().ffill()
    df_bat['Total_Charge_Time_s'] = df_bat['Total_Charge_Time_s'].bfill().ffill()
    
    # RUL calculation based on EOL capacity threshold (1.40 Ah = 70% of 2.0 Ah)
    eol_cycles = df_bat[df_bat['Discharge_Capacity_Ah'] <= eol_threshold]['Cycle_Index'].values
    if len(eol_cycles) > 0:
        eol_cycle = eol_cycles[0]
    else:
        eol_cycle = df_bat['Cycle_Index'].max()
        
    df_bat['RUL'] = np.maximum(0, eol_cycle - df_bat['Cycle_Index'])
    return df_bat

print("Extraction engine compiled successfully.")

## Module 2: Extraction Execution Across Battery Cells

In [ ]:
data_dir = "nasa_raw_data/FY08Q4"
batteries = ['B0005', 'B0006', 'B0007', 'B0018']
all_dfs = []

for bat_id in batteries:
    mat_path = os.path.join(data_dir, f"{bat_id}.mat")
    df_b = extract_battery_data(mat_path, bat_id)
    all_dfs.append(df_b)
    print(f"Cell {bat_id}: {len(df_b)} cycles extracted. Initial Cap: {df_b['Discharge_Capacity_Ah'].iloc[0]:.4f} Ah, Final Cap: {df_b['Discharge_Capacity_Ah'].iloc[-1]:.4f} Ah")

df_raw = pd.concat(all_dfs, ignore_index=True)
print(f"\nTotal Dataset Dimensions: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
display(df_raw.head())

### Capacity Degradation Trajectories and EOL Thresholds Across Cells

Plots measured discharge capacity across operating cycles for all 4 cells, highlighting the non-linear capacity fade and the 1.40 Ah (70% State of Health) End of Life (EOL) boundary.

In [ ]:
plt.figure(figsize=(14, 6))
colors = {"B0005": "navy", "B0006": "crimson", "B0007": "darkgreen", "B0018": "purple"}

for bat_id in batteries:
    sub_df = df_raw[df_raw["Battery_ID"] == bat_id]
    plt.plot(sub_df["Cycle_Index"], sub_df["Discharge_Capacity_Ah"], marker='o', markersize=3, label=f"Cell {bat_id}", color=colors[bat_id], linewidth=1.8)

plt.axhline(1.40, color="red", linestyle="--", linewidth=2, label="EOL Threshold (1.40 Ah / 70% Rated Capacity)")
plt.axhline(2.00, color="gray", linestyle=":", label="Nominal Rated Capacity (2.00 Ah)")
plt.title("NASA Battery Dataset: Discharge Capacity Degradation Curves Across Cells", fontsize=14, fontweight="bold")
plt.xlabel("Cycle Number")
plt.ylabel("Discharge Capacity (Ah)")
plt.legend(loc="upper right", fontsize=10)
plt.tight_layout()
plt.show()

### Electrochemical Impedance Growth and State of Health Dynamics

Contrasts State of Health (SOH %) against internal charge transfer resistance ($R_{ct}$) growth as active lithium is consumed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for bat_id in batteries:
    sub_df = df_raw[df_raw["Battery_ID"] == bat_id]
    axes[0].plot(sub_df["Cycle_Index"], sub_df["SOH_Pct"], label=f"Cell {bat_id}", color=colors[bat_id], linewidth=1.8)
    axes[1].plot(sub_df["Cycle_Index"], sub_df["Charge_Transfer_Resistance_Rct"], label=f"Cell {bat_id}", color=colors[bat_id], linewidth=1.8)

axes[0].axhline(70, color="red", linestyle="--", label="EOL Threshold (70% SOH)")
axes[0].set_title("State of Health (SOH %) Trajectory", fontweight="bold")
axes[0].set_xlabel("Cycle Number")
axes[0].set_ylabel("State of Health (%)")
axes[0].legend()

axes[1].set_title("Charge Transfer Resistance (Rct) Impedance Rise", fontweight="bold")
axes[1].set_xlabel("Cycle Number")
axes[1].set_ylabel("Rct Resistance (Ohms)")
axes[1].legend()

plt.suptitle("Electrochemical Degradation Dynamics: Capacity Loss vs. Impedance Rise", fontsize=15, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

### Inter-Feature Correlation Heatmap with Target RUL

Evaluates Pearson correlation coefficients ($r$) between extracted operational parameters and Remaining Useful Life.

In [ ]:
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns
corr_matrix = df_raw[numeric_cols].corr()

plt.figure(figsize=(14, 11))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5, cbar_kws={'label': 'Pearson Correlation (r)'})
plt.title("Correlation Heatmap of Extracted Physical Features vs. Target RUL", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Module 3: Feature Standardization and Dataset Export

Standardizes physical features into zero-mean, unit-variance distributions while preserving battery cell identifiers and target RUL.

In [ ]:
feature_cols = [
    'Discharge_Capacity_Ah', 'SOH_Pct', 'Max_Discharge_Voltage_V', 'Min_Discharge_Voltage_V',
    'Mean_Discharge_Voltage_V', 'Mean_Discharge_Current_A', 'Max_Discharge_Temp_C',
    'Mean_Discharge_Temp_C', 'Discharge_Duration_s', 'Time_to_3_5V_s', 'Time_to_3_2V_s',
    'CC_Charge_Time_s', 'CV_Charge_Time_s', 'Total_Charge_Time_s', 'Max_Charge_Temp_C',
    'Electrolyte_Resistance_Re', 'Charge_Transfer_Resistance_Rct'
]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(df_raw[feature_cols])

df_scaled = pd.DataFrame(scaled_features, columns=feature_cols)
df_scaled['Battery_ID'] = df_raw['Battery_ID']
df_scaled['Cycle_Index'] = df_raw['Cycle_Index']
df_scaled['RUL'] = df_raw['RUL']

cols_order = ['Battery_ID', 'Cycle_Index'] + feature_cols + ['RUL']
df_scaled = df_scaled[cols_order]

os.makedirs("dataset_nasa", exist_ok=True)
df_raw.to_csv("dataset_nasa/nasa_battery_raw_features.csv", index=False)
df_scaled.to_csv("dataset_nasa/nasa_battery_standard_scaled.csv", index=False)

print("Export Complete:")
print(f"1. Raw Physical Dataset:         dataset_nasa/nasa_battery_raw_features.csv (Shape: {df_raw.shape})")
print(f"2. Standard Scaled Dataset:      dataset_nasa/nasa_battery_standard_scaled.csv (Shape: {df_scaled.shape})")
print(f"Null Values:                     {df_scaled.isnull().sum().sum()}")
df_scaled.head()